**Token Replay after Recovery.** This figure measures the real LLM work that must be repeated after three recovery policies lose different amounts of valid work. AgentTX causal rollback keeps both valid documents and therefore invokes no replay model call; an optimistic temporal checkpoint regenerates one document; whole-branch abort regenerates two. All curves use the same DeepSeek model and the same AgentTX filesystem/dependency substrate.

In [1]:
# ipython -c "%run plot_token_recovery.ipynb"

import os
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

STANDARD_WIDTH = 17.8

def cm_to_inch(value):
    return value / 2.54

plt.rcParams.update(plt.rcParamsDefault)
matplotlib.rcParams['text.usetex'] = False
plt.rcParams['font.family'] = 'Nimbus Roman'
plt.rcParams['axes.grid'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['ytick.direction'] = 'in'
plt.rcParams['legend.frameon'] = True
plt.rcParams['legend.edgecolor'] = '0.55'
plt.rcParams['legend.framealpha'] = 1.0
plt.rcParams['legend.fancybox'] = False

STYLES = {
    'causal': dict(color='#c00000', marker='s', linestyle='-', linewidth=1.0, markersize=3.2),
    'temporal_checkpoint': dict(color='#e78129', marker='x', linestyle=':', linewidth=0.9, markersize=3.6, markeredgewidth=0.9),
    'whole_branch_abort': dict(color='black', marker='o', linestyle='--', linewidth=0.8, markersize=2.8, markerfacecolor='none'),
}
LABELS = {
    'causal': 'AgentTX causal (ours)',
    'temporal_checkpoint': 'optimistic checkpoint',
    'whole_branch_abort': 'whole-branch abort',
}
MODES = list(STYLES)

cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == 'motivation' else cwd
RESULTS = ROOT / 'experiments' / 'results'
FIGDIR = ROOT / 'motivation'

def provider_result(name):
    """Prefer the default provider's result dir, then any provider dir,
    then the legacy top-level result file."""
    results = ROOT / 'experiments' / 'results'
    provider = os.environ.get('AGENTTX_PROVIDER', 'deepseek')
    preferred = results / provider / name
    if preferred.exists():
        return preferred
    directories = sorted(
        path for path in results.iterdir()
        if path.is_dir() and (path / name).exists()
    )
    if directories:
        return directories[0] / name
    return results / name
df = pd.read_csv(provider_result('token_recovery.csv'))
numeric = [
    'document_lines', 'total_tokens_mean', 'completion_tokens_mean',
    'regenerated_documents_mean', 'wall_s_p95', 'success_rate', 'host_leak_rate',
]
df[numeric] = df[numeric].apply(pd.to_numeric)

def series(mode, metric):
    rows = df[df['mode'] == mode].sort_values('document_lines')
    return rows['document_lines'].to_numpy(), rows[metric].to_numpy(dtype=float)

fig = plt.figure(dpi=300, figsize=(cm_to_inch(STANDARD_WIDTH), cm_to_inch(7.0)))
handles = []
panels = [
    ('total_tokens_mean', 'Replay tokens', '(a) Total LLM work'),
    ('completion_tokens_mean', 'Completion tokens', '(b) Regenerated content'),
    ('regenerated_documents_mean', 'Documents regenerated', '(c) Valid work lost'),
    ('wall_s_p95', 'Recovery p95 (s)', '(d) Replay latency'),
]
for index, (metric, ylabel, subtitle) in enumerate(panels, start=1):
    ax = plt.subplot(2, 2, index)
    for mode in MODES:
        x, y = series(mode, metric)
        handle, = ax.plot(x, y, **STYLES[mode], label=LABELS[mode])
        if index == 1:
            handles.append(handle)
    ax.set_ylabel(ylabel, fontsize=8)
    ax.set_xlabel(f'Document size (# entries)\n{subtitle}', fontsize=7)
    ax.set_xticks(sorted(df['document_lines'].unique()))
    ax.tick_params(axis='both', labelsize=7)
    ax.set_ylim(bottom=-0.03 * max(1.0, ax.get_ylim()[1]))

ax_tokens = fig.axes[0]
x_abort, y_abort = series('whole_branch_abort', 'total_tokens_mean')
ax_tokens.annotate(
    f'{y_abort[-1]:,.0f} tokens avoided',
    xy=(x_abort[-1], y_abort[-1]), xytext=(-72, -15), textcoords='offset points',
    fontsize=6.5, color='#c00000',
    arrowprops=dict(arrowstyle='-', color='#c00000', linewidth=0.6),
)

fig.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, 1.035), ncol=3,
           fontsize=7, columnspacing=1.0, handlelength=1.8, handletextpad=0.35, borderpad=0.3)
plt.tight_layout(pad=0.6, h_pad=1.6, w_pad=1.2, rect=[0.0, 0.0, 1.0, 0.91])
plt.savefig(FIGDIR / 'FIG-Token-Recovery.pdf', bbox_inches='tight', pad_inches=0.02,
            metadata={'CreationDate': None, 'ModDate': None})
plt.savefig(FIGDIR / 'FIG-Token-Recovery.png', dpi=300, bbox_inches='tight', pad_inches=0.02)
plt.show()

largest = df[df['document_lines'] == df['document_lines'].max()].set_index('mode')
for mode in MODES:
    row = largest.loc[mode]
    print(f"{LABELS[mode]:24s}: replay={row['total_tokens_mean']:.0f} tokens, documents={row['regenerated_documents_mean']:.0f}, p95={row['wall_s_p95']:.2f}s")
assert (df['success_rate'] == 1.0).all()
assert (df['host_leak_rate'] == 0.0).all()


AgentTX causal (ours)   : replay=0 tokens, documents=0, p95=1.94s
optimistic checkpoint   : replay=1425 tokens, documents=1, p95=21.21s
whole-branch abort      : replay=3340 tokens, documents=2, p95=36.23s
